In [1]:
# step1_preprocess.py
import pandas as pd
import numpy as np
import monpa
import time
import multiprocessing
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings('ignore')

# 定義斷詞 Worker
def monpa_worker(texts):
    results = []
    for text in texts:
        if pd.isna(text) or str(text).strip() == "":
            results.append("")
            continue
        try:
            words = [item for item in monpa.cut(str(text))]
            # 過濾掉長度為 0 的空字串
            words = [w for w in words if len(w.strip()) > 0]
            results.append(" ".join(words))
        except Exception:
            results.append("")
    return results

if __name__ == '__main__':
    print("🚀 [步驟一] Monpa 斷詞預處理開始...")
    start_time = time.time()
    num_cores = multiprocessing.cpu_count()

    # 1. 讀取原始資料
    try:
        df_train = pd.read_csv('TrainingData.csv')
        df_test = pd.read_csv('TestingData.csv')
        print(f"   - 訓練集: {len(df_train)} 筆")
        print(f"   - 測試集: {len(df_test)} 筆")
    except:
        print("❌ 錯誤：找不到資料集檔案。")
        exit()

    # 合併資料
    df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    df_all['raw_text'] = df_all['Column2'].fillna('') + " " + df_all['Column3'].fillna('')
    raw_texts = df_all['raw_text'].values
    
    # 2. 多核斷詞
    print(f"⚡️ 正在進行斷詞 (使用 {num_cores} 核心)...")
    chunks = np.array_split(raw_texts, num_cores)
    
    with Parallel(n_jobs=num_cores, backend='loky', verbose=5) as parallel:
        processed_chunks = parallel(delayed(monpa_worker)(chunk) for chunk in chunks)
    
    # 3. 儲存結果
    df_all['text_segmented'] = np.concatenate(processed_chunks)
    
    # 只存需要的欄位，保持檔案輕量
    output_file = 'Processed_Data.csv'
    df_all[['Column1', 'text_segmented']].to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 斷詞完成！已存檔至: {output_file}")
    print(f"⏱️ 總耗時: {(time.time() - start_time)/60:.1f} 分鐘")

+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
🚀 [步驟一] Monpa 斷詞預處理開始...
   - 訓練集: 11671 筆
   - 測試集: 35604 筆
⚡️ 正在進行斷詞 (使用 12 核心)...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   4 out of  12 | elapsed:  2.8min remaining:  5.5min
[Parallel(n_jobs=12)]: Done   7 out of  12 | elapsed:  2.8min remaining:  2.0min
[Parallel(n_jobs=12)]: Done  10 out of  12 | elapsed:  2.8min remaining:   34.0s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:  2.9min finished


✅ 斷詞完成！已存檔至: Processed_Data.csv
⏱️ 總耗時: 2.9 分鐘
